# Dataset Analysis

In [1]:
import anndata as ad
import scanpy as sc
from skopt.space import Real, Categorical
import pandas as pd

# Load training data
adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)


# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)


# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=36473)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.X#.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['Manually_curated_celltype']

X_test = adata_test.X#.to_df()
gene_names_train = adata_train.var_names
y_test = adata_test.obs['Manually_curated_celltype']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'scrublet'
    obsm: 'X_umap'
Before filtering highly variable genes ---
AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'D

/tmp/ipykernel_7111/583758881.py:49: UserWarning: `n_top_genes` (=36473) > number of normalized dispersions (=27060), returning all genes with normalized dispersions.
  sc.pp.highly_variable_genes(adata, n_top_genes=36473)


After filtering highly variable genes ---
AnnData object with n_obs × n_vars = 27620 × 27060
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'scrublet', 'log1p', 'hvg'
    obsm: 'X

In [2]:
classes = adata.obs['Manually_curated_celltype']

classes_series = pd.Series(classes)

min_samples = 5
class_counts = classes_series.value_counts()
print(class_counts)

Manually_curated_celltype
Tnaive/CM_CD4               6609
Classical monocytes         5850
Teffector/EM_CD4            4331
NK_CD16+                    3539
Tem/emra_CD8                1983
Tnaive/CM_CD8               1182
Tregs                        798
Trm/em_CD8                   635
NK_CD56bright_CD16-          445
T_CD4/CD8                    270
Nonclassical monocytes       236
MNP/T doublets               229
Cycling T&NK                 215
Naive B cells                194
Memory B cells               171
Megakaryocytes               163
Tfh                          111
MAIT                         107
DC2                          100
Plasma cells                  66
T/B doublets                  62
Progenitor                    61
Tgd_CRTAM+                    47
pDC                           43
Trm_Th1/Th17                  35
Plasmablasts                  35
Mast cells                    22
ABCs                          21
ILC3                          15
DC1              

Cells remaining after filtering min_samples=150
Tnaive/CM_CD4               6609
Classical monocytes         5850
Teffector/EM_CD4            4331
NK_CD16+                    3539
Tem/emra_CD8                1983
Tnaive/CM_CD8               1182
Tregs                        798
Trm/em_CD8                   635
NK_CD56bright_CD16-          445
T_CD4/CD8                    270
Nonclassical monocytes       236
MNP/T doublets               229
Cycling T&NK                 215
Naive B cells                194
Memory B cells               171
Megakaryocytes               163

In [3]:
classes = adata.obs['scumi-annotation']

classes_series = pd.Series(classes)

min_samples = 5
class_counts = classes_series.value_counts()
print(class_counts)

scumi-annotation
CD4+ T cell            13393
CD14+ monocyte          6481
Natural killer cell     4087
Cytotoxic T cell        2704
B cell                   395
Megakaryocyte            269
Dendritic cell           185
Plasma cell              106
Name: count, dtype: int64


Missing cells from marker genes:
CD16+ monocyte
Plasmacytoid dendritic cell